In [1]:
list.of.packages <- c("haven","labelled","tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=FALSE)  

── Attaching core tidyverse packages ────────────────────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ──────────────────────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transp

In [2]:
path <- "../IA_2019-21_DHS/IAMR7EDT_Men_Stata/IAMR7EFL.DTA"

### Select columns of interest

In [3]:
df_men <- haven::read_dta(path,
                            col_select=c(# Identifiers
                                         "mcaseid","mv001","mv002","mv003","mv006","mv007","mv016","mv008",
                                         # Man characteristics
                                         "mv012","mv013","mv130","sm118","mv133","mv463z","mv481","mv717",
                                         # Other health outcomes
                                         "smb18s","smb18d","smb74",
                                         # Nutrition
                                         "sm630a","sm630b","sm630c","sm630d","sm630e","sm630f","sm630g","sm630h","sm630i",
                                         # Wealth
                                         "mv190",
                                         # Residence
                                         "mv025","mv135"
                                       )
                            )
sprintf("%i x %i dataframe", nrow(df_men), ncol(df_men))
head(df_men,2)

[1] "101839 x 31 dataframe"

mcaseid,mv001,mv002,mv003,mv006,mv007,mv008,mv012,mv013,mv016,mv025,mv130,mv133,mv135,mv190,mv463z,mv481,mv717,sm118,sm630a,sm630b,sm630c,sm630d,sm630e,sm630f,sm630g,sm630h,sm630i,smb18s,smb18d,smb74
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>
0100103080 5,130,80,5,12,2019,1440,24,2,2,2,2,8,1,2,1,0,0,4,2,3,2,2,3,3,2,3,3,134,91,101
0100103080 6,130,80,6,12,2019,1440,21,2,2,2,2,4,1,2,1,0,6,4,2,3,2,3,2,3,2,2,2,105,70,105


# Creation of variables after reading labels

In [4]:
df_men_var <- df_men %>%
                    mutate(DHS_round="2019", 
                           # Man characteristics
                           Current_age = mv012,
                           Age_group=haven::as_factor(mv013),
                           Religion_hindu = ifelse(mv130==1,1,0), 
                           Religion_muslim = ifelse(mv130==2,1,0), 
                           Religion_not_hindu_nor_muslim = ifelse(mv130!=1 & mv130!=2,1,0),
                           Ethni_SC =ifelse(sm118==1,1,
                                                  ifelse(sm118==8|is.na(sm118),NA,0)),
                           Ethni_ST =ifelse(sm118==2,1,
                                                        ifelse(sm118==8|is.na(sm118),NA,0)),
                           Ethni_OBC =ifelse(sm118==3,1,
                                                        ifelse(sm118==8|is.na(sm118),NA,0)),
                           Ethni_other =ifelse(sm118==4,1,
                                                        ifelse(sm118==8|is.na(sm118),NA,0)),
                           N_year_educ = ifelse(mv133==97,NA,mv133),
                           Smoker = ifelse(mv463z==0,1,0),
                           Health_insurance = mv481,
                           Profession = case_when(mv717 == 0 ~ "no_work",
                                                  mv717 == 1 ~ "prof_tech_manag",
                                                  mv717 == 3 ~ "clerical",
                                                  mv717 == 4 ~ "sales",
                                                  mv717 == 6 ~ "agri",
                                                  mv717 == 5 ~ "services",
                                                  mv717 == 7 ~ "manual",
                                                  TRUE ~ NA),
                           # Other health outcomes
                           Blood_pressure_systo = ifelse(smb18s %in% c(994,995,996),NA, smb18s),
                           Blood_pressure_diasto = ifelse(smb18d %in% c(994,995,996),NA, smb18d),
                           Blood_glucose_level= ifelse(smb74 %in% c(995,996,998),NA,smb74),
                           # Nutrition
                           Diet_milk_curd_daily = ifelse(sm630a==1,1,0),
                           Diet_milk_curd_week = ifelse(sm630a==2,1,0),
                           Diet_milk_curd_oc = ifelse(sm630a==3,1,0),
                           Diet_milk_curd_never = ifelse(sm630a==0,1,0),
                           Diet_pulses_beans_daily = ifelse(sm630b==1,1,0),
                           Diet_pulses_beans_week = ifelse(sm630b==2,1,0),
                           Diet_pulses_beans_oc = ifelse(sm630b==3,1,0),
                           Diet_pulses_beans_never = ifelse(sm630b==0,1,0),
                           Diet_green_veg_daily = ifelse(sm630c==1,1,0),
                           Diet_green_veg_week = ifelse(sm630c==2,1,0),
                           Diet_green_veg_oc = ifelse(sm630c==3,1,0),
                           Diet_green_veg_never = ifelse(sm630c==0,1,0),
                           Diet_fruits_daily = ifelse(sm630d==1,1,0),
                           Diet_fruits_week = ifelse(sm630d==2,1,0),
                           Diet_fruits_oc = ifelse(sm630d==3,1,0),
                           Diet_fruits_never = ifelse(sm630d==0,1,0),
                           Diet_eggs_daily = ifelse(sm630e==1,1,0),
                           Diet_eggs_week = ifelse(sm630e==2,1,0),
                           Diet_eggs_oc = ifelse(sm630e==3,1,0),
                           Diet_eggs_never = ifelse(sm630e==0,1,0),                           
                           Diet_fish_daily = ifelse(sm630f==1,1,0),
                           Diet_fish_week = ifelse(sm630f==2,1,0),
                           Diet_fish_oc = ifelse(sm630f==3,1,0),
                           Diet_fish_never = ifelse(sm630f==0,1,0), 
                           Diet_chicken_meat_daily = ifelse(sm630g==1,1,0),
                           Diet_chicken_meat_week = ifelse(sm630g==2,1,0),
                           Diet_chicken_meat_oc = ifelse(sm630g==3,1,0),
                           Diet_chicken_meat_never = ifelse(sm630g==0,1,0),  
                           Diet_fried_food_daily = ifelse(sm630h==1,1,0),
                           Diet_fried_food_week = ifelse(sm630h==2,1,0),
                           Diet_fried_food_oc = ifelse(sm630h==3,1,0),
                           Diet_fried_food_never = ifelse(sm630h==0,1,0), 
                           Diet_aerated_drinks_daily = ifelse(sm630i==1,1,0),
                           Diet_aerated_drinks_week = ifelse(sm630i==2,1,0),
                           Diet_aerated_drinks_oc = ifelse(sm630i==3,1,0),
                           Diet_aerated_drinks_never = ifelse(sm630i==0,1,0),
                           # Wealth
                           Wealth_lowest = ifelse(mv190==1,1,0),
                           Wealth_second = ifelse(mv190==2,1,0),
                           Wealth_middle = ifelse(mv190==3,1,0),
                           Wealth_fourth = ifelse(mv190==4,1,0),
                           Wealth_highest = ifelse(mv190==5,1,0),
                           # Residence
                           Urban = ifelse(mv025==1,1,0),
                           Usual_resident = ifelse(mv135==2,0,mv135)                            
)

## Select variables of interest

In [5]:
df_men_select <- df_men_var %>%
                        select(# IDs
                               DHS_round,DHSCLUST=mv001,HH_ID=mv002,Men_id=mcaseid,Resp_nb=mv003,
                               Interview_month=mv006,Interview_year=mv007,Interview_day=mv016,
                               # Man characteristics
                               Current_age, Age_group,
                               Religion_hindu, Religion_muslim, Religion_not_hindu_nor_muslim,
                               Ethni_SC,Ethni_ST,Ethni_OBC,Ethni_other,
                               N_year_educ,
                               Smoker,Health_insurance,
                               Profession,
                               # Other health outcomes
                               Blood_pressure_systo,Blood_pressure_diasto,Blood_glucose_level,
                               # Nutrition
                               Diet_milk_curd_daily,Diet_milk_curd_week,Diet_milk_curd_oc,Diet_milk_curd_never,
                               Diet_pulses_beans_daily,Diet_pulses_beans_week,Diet_pulses_beans_oc,Diet_pulses_beans_never,
                               Diet_green_veg_daily,Diet_green_veg_week,Diet_green_veg_oc,Diet_green_veg_never,
                               Diet_fruits_daily,Diet_fruits_week,Diet_fruits_oc,Diet_fruits_never,
                               Diet_eggs_daily,Diet_eggs_week,Diet_eggs_oc,Diet_eggs_never,                           
                               Diet_fish_daily,Diet_fish_week,Diet_fish_oc,Diet_fish_never,
                               Diet_chicken_meat_daily,Diet_chicken_meat_week,Diet_chicken_meat_oc,Diet_chicken_meat_never,  
                               Diet_fried_food_daily,Diet_fried_food_week,Diet_fried_food_oc,Diet_fried_food_never, 
                               Diet_aerated_drinks_daily,Diet_aerated_drinks_week,Diet_aerated_drinks_oc,Diet_aerated_drinks_never,
                               # Wealth
                               Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,
                               # Residence
                               Urban,Usual_resident
                                )
sprintf("%i x %i dataframe", nrow(df_men_select), ncol(df_men_select))
head(df_men_select, 2)

[1] "101839 x 67 dataframe"

DHS_round,DHSCLUST,HH_ID,Men_id,Resp_nb,Interview_month,Interview_year,Interview_day,Current_age,Age_group,Religion_hindu,Religion_muslim,Religion_not_hindu_nor_muslim,Ethni_SC,Ethni_ST,Ethni_OBC,Ethni_other,N_year_educ,Smoker,Health_insurance,Profession,Blood_pressure_systo,Blood_pressure_diasto,Blood_glucose_level,Diet_milk_curd_daily,Diet_milk_curd_week,Diet_milk_curd_oc,Diet_milk_curd_never,Diet_pulses_beans_daily,Diet_pulses_beans_week,Diet_pulses_beans_oc,Diet_pulses_beans_never,Diet_green_veg_daily,Diet_green_veg_week,Diet_green_veg_oc,Diet_green_veg_never,Diet_fruits_daily,Diet_fruits_week,Diet_fruits_oc,Diet_fruits_never,Diet_eggs_daily,Diet_eggs_week,Diet_eggs_oc,Diet_eggs_never,Diet_fish_daily,Diet_fish_week,Diet_fish_oc,Diet_fish_never,Diet_chicken_meat_daily,Diet_chicken_meat_week,Diet_chicken_meat_oc,Diet_chicken_meat_never,Diet_fried_food_daily,Diet_fried_food_week,Diet_fried_food_oc,Diet_fried_food_never,Diet_aerated_drinks_daily,Diet_aerated_drinks_week,Diet_aerated_drinks_oc,Diet_aerated_drinks_never,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident
<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2019,130,80,0100103080 5,5,12,2019,2,24,20-24,0,1,0,0,0,0,1,8,0,0,no_work,134,91,101,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1
2019,130,80,0100103080 6,6,12,2019,2,21,20-24,0,1,0,0,0,0,1,4,0,0,agri,105,70,105,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1


## Add hemoglobin and anemia levels in HH members file

In [6]:
path_HH <- "../IA_2019-21_DHS/IAPR7EDT_HHmembers_Stata/IAPR7EFL.DTA"
df_HH <- haven::read_dta(path_HH,
                         col_select=c("hvidx","hb0","hv001","hv002","hv003",
                                      # Hemoglobin level
                                      "hb56",
                                      # Body Mass index
                                      "hb40",
                                      # Bednet
                                      "hml12",
                                      # Water
                                      "hv204",
                                      # Air conditioner/cooler
                                      "sh50q",
                                      # WASH
                                      "hv237","hv205","hv201"
                                     )
                        )
sprintf("%i x %i dataframe", nrow(df_HH), ncol(df_HH))
head(df_HH, 2)

[1] "2843917 x 13 dataframe"

hvidx,hv001,hv002,hv003,hv201,hv204,hv205,hv237,sh50q,hb0,hb40,hb56,hml12
<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>,<dbl>,<dbl+lbl>,<dbl+lbl>,<dbl+lbl>
1,113,5,1,14,996,12,0,0,1,NA,NA(a),0
2,113,5,1,14,996,12,0,0,NA,NA,NA,0


In [7]:
df_hemoglo <- df_HH %>%
                mutate(Blood_hemo_level_alti = ifelse(hb56==997,NA,hb56),
                       Body_mass_index = ifelse(hb40 < 1200 | hb40> 6000, NA, hb40/100),
                       Bednet_slept = ifelse(hml12==0,0,1),
                       HH_time_water_source = ifelse(hv204==996,0,
                                                    ifelse(hv204==998,NA,hv204)),
                       HH_air_conditioner = sh50q,
                       HH_treat_water = ifelse(hv237==8,NA,hv237),
                       HH_no_toilet = ifelse(hv205==30|hv205==31,1,0),
                       HH_well_water = ifelse(hv201==21|hv201==31|hv201==32,1,ifelse(!is.na(hv201),0,NA))
                       )%>%
                select(hvidx,hv001,hv002,
                       Blood_hemo_level_alti,
                       Body_mass_index,
                       Bednet_slept,
                       HH_time_water_source,
                       HH_air_conditioner,
                       HH_treat_water,HH_no_toilet,HH_well_water)
sprintf("%i x %i dataframe", nrow(df_hemoglo), ncol(df_hemoglo))
head(df_hemoglo, 2)

[1] "2843917 x 11 dataframe"

hvidx,hv001,hv002,Blood_hemo_level_alti,Body_mass_index,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
1,113,5,NA,NA,0,0,0,0,0,0
2,113,5,NA,NA,0,0,0,0,0,0


In [8]:
df_men_select_hemo <- df_men_select %>%
                            left_join(df_hemoglo,by=c("DHSCLUST"="hv001","HH_ID"="hv002","Resp_nb"="hvidx"))
sprintf("%i x %i dataframe", nrow(df_men_select_hemo), ncol(df_men_select_hemo))
head(df_men_select_hemo, 2)

[1] "101839 x 75 dataframe"

DHS_round,DHSCLUST,HH_ID,Men_id,Resp_nb,Interview_month,Interview_year,Interview_day,Current_age,Age_group,Religion_hindu,Religion_muslim,Religion_not_hindu_nor_muslim,Ethni_SC,Ethni_ST,Ethni_OBC,Ethni_other,N_year_educ,Smoker,Health_insurance,Profession,Blood_pressure_systo,Blood_pressure_diasto,Blood_glucose_level,Diet_milk_curd_daily,Diet_milk_curd_week,Diet_milk_curd_oc,Diet_milk_curd_never,Diet_pulses_beans_daily,Diet_pulses_beans_week,Diet_pulses_beans_oc,Diet_pulses_beans_never,Diet_green_veg_daily,Diet_green_veg_week,Diet_green_veg_oc,Diet_green_veg_never,Diet_fruits_daily,Diet_fruits_week,Diet_fruits_oc,Diet_fruits_never,Diet_eggs_daily,Diet_eggs_week,Diet_eggs_oc,Diet_eggs_never,Diet_fish_daily,Diet_fish_week,Diet_fish_oc,Diet_fish_never,Diet_chicken_meat_daily,Diet_chicken_meat_week,Diet_chicken_meat_oc,Diet_chicken_meat_never,Diet_fried_food_daily,Diet_fried_food_week,Diet_fried_food_oc,Diet_fried_food_never,Diet_aerated_drinks_daily,Diet_aerated_drinks_week,Diet_aerated_drinks_oc,Diet_aerated_drinks_never,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Blood_hemo_level_alti,Body_mass_index,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
<chr>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
2019,130,80,0100103080 5,5,12,2019,2,24,20-24,0,1,0,0,0,0,1,8,0,0,no_work,134,91,101,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,117,21.53,0,0,0,1,0,0
2019,130,80,0100103080 6,6,12,2019,2,21,20-24,0,1,0,0,0,0,1,4,0,0,agri,105,70,105,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,155,20.40,0,0,0,1,0,0


## Add state and district according to 2011 census

In [9]:
df_2011_census <- read_csv("./interm/Matching_DHSclusters_2019_census_2011.csv",show_col_types = FALSE)
sprintf("%i x %i dataframe", nrow(df_2011_census), ncol(df_2011_census))
head(df_2011_census, 2)

[1] "30037 x 3 dataframe"

DHSCLUST,State_2011,District_2011
<dbl>,<chr>,<chr>
102,Jammu & Kashmir,Kupwara
103,Jammu & Kashmir,Kupwara


In [10]:
df_men_census <- df_men_select_hemo %>%
                                left_join(df_2011_census,by="DHSCLUST")%>%
                                relocate(c(State_2011,District_2011),.after=DHSCLUST)
sprintf("%i x %i dataframe", nrow(df_men_census), ncol(df_men_census))
head(df_men_census, 2)

[1] "101839 x 77 dataframe"

DHS_round,DHSCLUST,State_2011,District_2011,HH_ID,Men_id,Resp_nb,Interview_month,Interview_year,Interview_day,Current_age,Age_group,Religion_hindu,Religion_muslim,Religion_not_hindu_nor_muslim,Ethni_SC,Ethni_ST,Ethni_OBC,Ethni_other,N_year_educ,Smoker,Health_insurance,Profession,Blood_pressure_systo,Blood_pressure_diasto,Blood_glucose_level,Diet_milk_curd_daily,Diet_milk_curd_week,Diet_milk_curd_oc,Diet_milk_curd_never,Diet_pulses_beans_daily,Diet_pulses_beans_week,Diet_pulses_beans_oc,Diet_pulses_beans_never,Diet_green_veg_daily,Diet_green_veg_week,Diet_green_veg_oc,Diet_green_veg_never,Diet_fruits_daily,Diet_fruits_week,Diet_fruits_oc,Diet_fruits_never,Diet_eggs_daily,Diet_eggs_week,Diet_eggs_oc,Diet_eggs_never,Diet_fish_daily,Diet_fish_week,Diet_fish_oc,Diet_fish_never,Diet_chicken_meat_daily,Diet_chicken_meat_week,Diet_chicken_meat_oc,Diet_chicken_meat_never,Diet_fried_food_daily,Diet_fried_food_week,Diet_fried_food_oc,Diet_fried_food_never,Diet_aerated_drinks_daily,Diet_aerated_drinks_week,Diet_aerated_drinks_oc,Diet_aerated_drinks_never,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Blood_hemo_level_alti,Body_mass_index,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water
<chr>,<dbl>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>
2019,130,Jammu & Kashmir,Kupwara,80,0100103080 5,5,12,2019,2,24,20-24,0,1,0,0,0,0,1,8,0,0,no_work,134,91,101,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,117,21.53,0,0,0,1,0,0
2019,130,Jammu & Kashmir,Kupwara,80,0100103080 6,6,12,2019,2,21,20-24,0,1,0,0,0,0,1,4,0,0,agri,105,70,105,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,155,20.40,0,0,0,1,0,0


## Add cells ID

In [11]:
df_cells_coord_01 <- read_csv("../matching_grid_cells_DHS_clusters/output/DHS_19_matching_clust_cells_0.1.csv",show_col_types = FALSE)
sprintf("%i x %i dataframe", nrow(df_cells_coord_01), ncol(df_cells_coord_01))
head(df_cells_coord_01, 2)

[1] "30079 x 3 dataframe"

DHSCLUST,cell_x,cell_y
<dbl>,<dbl>,<dbl>
101,73.8,34.4
102,74.3,34.4


In [12]:
df_cells_coord_025 <- read_csv("../matching_grid_cells_DHS_clusters/output/DHS_19_matching_clust_cells_0.25.csv",show_col_types = FALSE)
sprintf("%i x %i dataframe", nrow(df_cells_coord_025), ncol(df_cells_coord_025))
head(df_cells_coord_025, 2)

[1] "30079 x 3 dataframe"

DHSCLUST,cell_x,cell_y
<dbl>,<dbl>,<dbl>
101,73.75,34.5
102,74.25,34.5


In [13]:
df_men_cells <- df_men_census %>%
                                left_join(df_cells_coord_01%>%mutate(cell_x_01=cell_x,cell_y_01=cell_y),
                                          by="DHSCLUST")%>%
                                left_join(df_cells_coord_025%>%mutate(cell_x_025=cell_x,cell_y_025=cell_y),
                                          by="DHSCLUST")%>%
                                relocate(c(cell_x_01,cell_y_01,cell_x_025,cell_y_025),.after=District_2011)
sprintf("%i x %i dataframe", nrow(df_men_cells), ncol(df_men_cells))
head(df_men_cells, 2)

[1] "101839 x 85 dataframe"

DHS_round,DHSCLUST,State_2011,District_2011,cell_x_01,cell_y_01,cell_x_025,cell_y_025,HH_ID,Men_id,Resp_nb,Interview_month,Interview_year,Interview_day,Current_age,Age_group,Religion_hindu,Religion_muslim,Religion_not_hindu_nor_muslim,Ethni_SC,Ethni_ST,Ethni_OBC,Ethni_other,N_year_educ,Smoker,Health_insurance,Profession,Blood_pressure_systo,Blood_pressure_diasto,Blood_glucose_level,Diet_milk_curd_daily,Diet_milk_curd_week,Diet_milk_curd_oc,Diet_milk_curd_never,Diet_pulses_beans_daily,Diet_pulses_beans_week,Diet_pulses_beans_oc,Diet_pulses_beans_never,Diet_green_veg_daily,Diet_green_veg_week,Diet_green_veg_oc,Diet_green_veg_never,Diet_fruits_daily,Diet_fruits_week,Diet_fruits_oc,Diet_fruits_never,Diet_eggs_daily,Diet_eggs_week,Diet_eggs_oc,Diet_eggs_never,Diet_fish_daily,Diet_fish_week,Diet_fish_oc,Diet_fish_never,Diet_chicken_meat_daily,Diet_chicken_meat_week,Diet_chicken_meat_oc,Diet_chicken_meat_never,Diet_fried_food_daily,Diet_fried_food_week,Diet_fried_food_oc,Diet_fried_food_never,Diet_aerated_drinks_daily,Diet_aerated_drinks_week,Diet_aerated_drinks_oc,Diet_aerated_drinks_never,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Blood_hemo_level_alti,Body_mass_index,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water,cell_x.x,cell_y.x,cell_x.y,cell_y.y
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2019,130,Jammu & Kashmir,Kupwara,74.2,34.4,74.25,34.5,80,0100103080 5,5,12,2019,2,24,20-24,0,1,0,0,0,0,1,8,0,0,no_work,134,91,101,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,117,21.53,0,0,0,1,0,0,74.2,34.4,74.25,34.5
2019,130,Jammu & Kashmir,Kupwara,74.2,34.4,74.25,34.5,80,0100103080 6,6,12,2019,2,21,20-24,0,1,0,0,0,0,1,4,0,0,agri,105,70,105,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,155,20.40,0,0,0,1,0,0,74.2,34.4,74.25,34.5


## Convert interview date in CDC

In [14]:
df_men_CDC <- df_men_cells %>%
                        mutate(Measured_date = as.Date(paste(Interview_year, Interview_month, Interview_day, sep = "-")),
                               Measured_date_CDC = as.numeric(Measured_date - as.Date("1900-01-01")))%>%
                        relocate(c(Measured_date_CDC,Measured_date),.after=Interview_day)
sprintf("%i x %i dataframe", nrow(df_men_CDC), ncol(df_men_CDC))
head(df_men_CDC, 2)

[1] "101839 x 87 dataframe"

DHS_round,DHSCLUST,State_2011,District_2011,cell_x_01,cell_y_01,cell_x_025,cell_y_025,HH_ID,Men_id,Resp_nb,Interview_month,Interview_year,Interview_day,Measured_date_CDC,Measured_date,Current_age,Age_group,Religion_hindu,Religion_muslim,Religion_not_hindu_nor_muslim,Ethni_SC,Ethni_ST,Ethni_OBC,Ethni_other,N_year_educ,Smoker,Health_insurance,Profession,Blood_pressure_systo,Blood_pressure_diasto,Blood_glucose_level,Diet_milk_curd_daily,Diet_milk_curd_week,Diet_milk_curd_oc,Diet_milk_curd_never,Diet_pulses_beans_daily,Diet_pulses_beans_week,Diet_pulses_beans_oc,Diet_pulses_beans_never,Diet_green_veg_daily,Diet_green_veg_week,Diet_green_veg_oc,Diet_green_veg_never,Diet_fruits_daily,Diet_fruits_week,Diet_fruits_oc,Diet_fruits_never,Diet_eggs_daily,Diet_eggs_week,Diet_eggs_oc,Diet_eggs_never,Diet_fish_daily,Diet_fish_week,Diet_fish_oc,Diet_fish_never,Diet_chicken_meat_daily,Diet_chicken_meat_week,Diet_chicken_meat_oc,Diet_chicken_meat_never,Diet_fried_food_daily,Diet_fried_food_week,Diet_fried_food_oc,Diet_fried_food_never,Diet_aerated_drinks_daily,Diet_aerated_drinks_week,Diet_aerated_drinks_oc,Diet_aerated_drinks_never,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Blood_hemo_level_alti,Body_mass_index,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water,cell_x.x,cell_y.x,cell_x.y,cell_y.y
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<date>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2019,130,Jammu & Kashmir,Kupwara,74.2,34.4,74.25,34.5,80,0100103080 5,5,12,2019,2,43799,2019-12-02,24,20-24,0,1,0,0,0,0,1,8,0,0,no_work,134,91,101,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,117,21.53,0,0,0,1,0,0,74.2,34.4,74.25,34.5
2019,130,Jammu & Kashmir,Kupwara,74.2,34.4,74.25,34.5,80,0100103080 6,6,12,2019,2,43799,2019-12-02,21,20-24,0,1,0,0,0,0,1,4,0,0,agri,105,70,105,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,155,20.40,0,0,0,1,0,0,74.2,34.4,74.25,34.5


## Final selection

In [15]:
df_men_final <- df_men_CDC %>%
                            filter(!is.na(Measured_date_CDC) & !is.na(cell_x_01) & Interview_year == 2019)
sprintf("%i x %i dataframe", nrow(df_men_final), ncol(df_men_final))
head(df_men_final, 2)

[1] "47587 x 87 dataframe"

DHS_round,DHSCLUST,State_2011,District_2011,cell_x_01,cell_y_01,cell_x_025,cell_y_025,HH_ID,Men_id,Resp_nb,Interview_month,Interview_year,Interview_day,Measured_date_CDC,Measured_date,Current_age,Age_group,Religion_hindu,Religion_muslim,Religion_not_hindu_nor_muslim,Ethni_SC,Ethni_ST,Ethni_OBC,Ethni_other,N_year_educ,Smoker,Health_insurance,Profession,Blood_pressure_systo,Blood_pressure_diasto,Blood_glucose_level,Diet_milk_curd_daily,Diet_milk_curd_week,Diet_milk_curd_oc,Diet_milk_curd_never,Diet_pulses_beans_daily,Diet_pulses_beans_week,Diet_pulses_beans_oc,Diet_pulses_beans_never,Diet_green_veg_daily,Diet_green_veg_week,Diet_green_veg_oc,Diet_green_veg_never,Diet_fruits_daily,Diet_fruits_week,Diet_fruits_oc,Diet_fruits_never,Diet_eggs_daily,Diet_eggs_week,Diet_eggs_oc,Diet_eggs_never,Diet_fish_daily,Diet_fish_week,Diet_fish_oc,Diet_fish_never,Diet_chicken_meat_daily,Diet_chicken_meat_week,Diet_chicken_meat_oc,Diet_chicken_meat_never,Diet_fried_food_daily,Diet_fried_food_week,Diet_fried_food_oc,Diet_fried_food_never,Diet_aerated_drinks_daily,Diet_aerated_drinks_week,Diet_aerated_drinks_oc,Diet_aerated_drinks_never,Wealth_lowest,Wealth_second,Wealth_middle,Wealth_fourth,Wealth_highest,Urban,Usual_resident,Blood_hemo_level_alti,Body_mass_index,Bednet_slept,HH_time_water_source,HH_air_conditioner,HH_treat_water,HH_no_toilet,HH_well_water,cell_x.x,cell_y.x,cell_x.y,cell_y.y
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<date>,<dbl>,<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl+lbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2019,130,Jammu & Kashmir,Kupwara,74.2,34.4,74.25,34.5,80,0100103080 5,5,12,2019,2,43799,2019-12-02,24,20-24,0,1,0,0,0,0,1,8,0,0,no_work,134,91,101,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,117,21.53,0,0,0,1,0,0,74.2,34.4,74.25,34.5
2019,130,Jammu & Kashmir,Kupwara,74.2,34.4,74.25,34.5,80,0100103080 6,6,12,2019,2,43799,2019-12-02,21,20-24,0,1,0,0,0,0,1,4,0,0,agri,105,70,105,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,155,20.40,0,0,0,1,0,0,74.2,34.4,74.25,34.5


## SAVE

In [16]:
saveRDS(df_men_final,"./output/df_men_2019.rds")